In [1]:
!pip -q install earthengine-api geemap geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.5 MB/s eta 0:00:00


In [16]:
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
from datetime import date

In [17]:
ee.Authenticate()
ee.Initialize(project="sundarbans-ai")

print("Google Earth Engine Connected")

Google Earth Engine Connected


In [18]:
# Study Area Load
sundarbans = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("NAME", "Sundarbans Reserved Forest")
)

# 2. Maximum Available Temporal Bounds (From FIRMS inception to TODAY)
START_DATE = "2015-01-01"
END_DATE = date.today().strftime("%Y-%m-%d")

print(f"Full Historical Study Window: {START_DATE} to {END_DATE}")

# Load NASA FIRMS Active Fire Dataset
firms = ee.ImageCollection("FIRMS").filterBounds(sundarbans).filterDate(START_DATE, END_DATE)

# Base raster (SRTM DEM) to keep spatial projection and geometry intact
base_raster = ee.Image("USGS/SRTMGL1_003").clip(sundarbans)

# Create Fire Mask (Composite of all fires over the last ~26 years)
fire_image = firms.select("T21").max().clip(sundarbans)
fire_mask = fire_image.gt(0).reproject(crs=base_raster.projection(), scale=500)

print("Maximum Historical FIRMS Data (2000-Present) Processed Successfully.")

Full Historical Study Window: 2015-01-01 to 2026-07-05
Maximum Historical FIRMS Data (2000-Present) Processed Successfully.


In [19]:
# Load NASA FIRMS Active Fire dataset
firms = ee.ImageCollection("FIRMS").filterBounds(sundarbans).filterDate(START_DATE, END_DATE)

# Select the 'T21' band (Bright temperature) or confidence image
# Reduce the collection to get locations with active fire alerts
fire_mask = firms.select("T21").max().gt(0).clip(sundarbans)

print("NASA FIRMS dataset filtered over Sundarbans.")

NASA FIRMS dataset filtered over Sundarbans.


In [20]:
# Convert fire pixels into vector points (Positive Samples)
fire_points = fire_mask.selfMask().sample(
    region=sundarbans,
    scale=500,  # FIRMS resolution scale
    numPixels=2000,
    seed=42,
    geometries=True
)

# Add label = 1 for Fire points
def set_fire_label(feature):
    return feature.set("fire_label", 1)

fire_labeled = fire_points.map(set_fire_label)

fire_count = fire_labeled.size().getInfo()
print(f"Extracted Positive Fire Samples (Label = 1): {fire_count}")

Extracted Positive Fire Samples (Label = 1): 1


In [21]:
# Invert the fire mask to get non-fire areas
non_fire_mask = fire_mask.unmask().eq(0).clip(sundarbans)

# Sample random points from non-fire areas (Negative Samples)
# We balance the dataset by sampling an equal or slightly higher number of negative points
non_fire_points = non_fire_mask.selfMask().sample(
    region=sundarbans,
    scale=500,
    numPixels=fire_count * 2,  # 1:2 ratio for robust training
    seed=101,
    geometries=True
)

# Add label = 0 for Non-Fire points
def set_non_fire_label(feature):
    return feature.set("fire_label", 0)

non_fire_labeled = non_fire_points.map(set_non_fire_label)

non_fire_count = non_fire_labeled.size().getInfo()
print(f"Extracted Negative Non-Fire Samples (Label = 0): {non_fire_count}")

Extracted Negative Non-Fire Samples (Label = 0): 2


In [22]:
# Merge positive and negative samples
final_dataset_points = fire_labeled.merge(non_fire_labeled)

# Visualize on Interactive Map
Map = geemap.Map()
Map.centerObject(sundarbans, 10)

# Add layers
Map.addLayer(sundarbans, {"color": "green"}, "Sundarbans Boundary")
Map.addLayer(non_fire_labeled, {"color": "blue"}, "Non-Fire Points (0)")
Map.addLayer(fire_labeled, {"color": "red"}, "Fire Points (1)")

Map

Map(center=[22.06671057154965, 89.47779532862853], controls=(WidgetControl(options=['position', 'transparent_b…

In [23]:
# 1. Load Sundarbans Boundary
sundarbans_core = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("NAME", "Sundarbans Reserved Forest")
)

# 2. Scientific Innovation: Create a 10km Buffer around Sundarbans to capture Edge Fire Risks
sundarbans_buffer = sundarbans_core.geometry().buffer(10000) # 10,000 meters = 10 km

# 3. Time Window (2012 to Present gives optimal balance between sample size and satellite data)
START_DATE = "2012-01-01"
END_DATE = date.today().strftime("%Y-%m-%d")

print(f"Study Window: {START_DATE} to {END_DATE}")
print("Strategy: Capturing fire risks within 10km buffer zone.")

# Load NASA FIRMS Active Fire Dataset
firms = ee.ImageCollection("FIRMS").filterBounds(sundarbans_buffer).filterDate(START_DATE, END_DATE)

# Base raster for consistent projection
base_raster = ee.Image("USGS/SRTMGL1_003").clip(sundarbans_buffer)

# Create Fire Mask inside the Buffer Zone
fire_image = firms.select("T21").max().clip(sundarbans_buffer)
fire_mask = fire_image.gt(0).reproject(crs=base_raster.projection(), scale=500)

# Extract Positive Fire Samples (Label = 1)
fire_points = fire_mask.selfMask().sample(
    region=sundarbans_buffer,
    scale=500,
    numPixels=2000,
    seed=42,
    geometries=True
)

def set_fire_label(feature):
    return feature.set("fire_label", 1)

fire_labeled = fire_points.map(set_fire_label)
fire_count = fire_labeled.size().getInfo()

print(f"Extracted Positive Fire Samples (Label = 1): {fire_count}")

# Extract Negative Non-Fire Samples (Label = 0) strictly from Core Forest & Buffer
non_fire_mask = fire_mask.unmask(0).eq(0).clip(sundarbans_buffer)

non_fire_points = non_fire_mask.selfMask().sample(
    region=sundarbans_buffer,
    scale=500,
    numPixels=max(fire_count * 2, 400), # 1:2 ratio or at least 400 points
    seed=101,
    geometries=True
)

def set_non_fire_label(feature):
    return feature.set("fire_label", 0)

non_fire_labeled = non_fire_points.map(set_non_fire_label)
non_fire_count = non_fire_labeled.size().getInfo()

print(f"Extracted Negative Non-Fire Samples (Label = 0): {non_fire_count}")

# Merge & Visualize
final_dataset_points = fire_labeled.merge(non_fire_labeled)

Map = geemap.Map()
Map.centerObject(sundarbans_core, 9)
Map.addLayer(sundarbans_buffer, {"color": "yellow"}, "10km Risk Buffer Zone", opacity=0.3)
Map.addLayer(sundarbans_core, {"color": "green"}, "Core Sundarbans Forest")
Map.addLayer(non_fire_labeled, {"color": "blue"}, "Non-Fire Points (0)")
Map.addLayer(fire_labeled, {"color": "red"}, "Fire Points (1)")

Map

Study Window: 2012-01-01 to 2026-07-05
Strategy: Capturing fire risks within 10km buffer zone.
Extracted Positive Fire Samples (Label = 1): 12
Extracted Negative Non-Fire Samples (Label = 0): 400


Map(center=[22.06671057154959, 89.4777953286285], controls=(WidgetControl(options=['position', 'transparent_bg…

In [24]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
# Exact Folder Path
SAVE_DIR = '/content/drive/My Drive/GeoAI/GeoAI-Based Forest Fire Detection and Vegetation Stress Monitoring Using Multi-Source Remote Sensing Data/02_Data'
os.makedirs(SAVE_DIR, exist_ok=True)

# Instant Save using geemap
csv_path = os.path.join(SAVE_DIR, 'Sundarbans_Fire_Labels_10km_Buffer.csv')
geemap.ee_to_csv(final_dataset_points, filename=csv_path)

print(f"File Saved:\n{csv_path}")

File Saved:
/content/drive/My Drive/GeoAI/GeoAI-Based Forest Fire Detection and Vegetation Stress Monitoring Using Multi-Source Remote Sensing Data/02_Data/Sundarbans_Fire_Labels_10km_Buffer.csv
